In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import cvxpy as cp
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
import os
import time
from googleapiclient.discovery import build
from google.oauth2 import service_account
from googleapiclient.http import MediaFileUpload

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
credentials_path = '/content/recommendationdrive.json'

if not os.path.exists(credentials_path):
    raise FileNotFoundError("Credentials file not found. Please upload the credentials.json file.")

credentials = service_account.Credentials.from_service_account_file(credentials_path, scopes=['https://www.googleapis.com/auth/drive'])

drive_service = build('drive', 'v3', credentials=credentials)

def check_for_new_file(folder_id, processed_files):
    query = f"'{folder_id}' in parents and trashed = false"

    response = drive_service.files().list(q=query).execute()
    files = response.get('files', [])

    new_files = [file for file in files if file['name'] not in processed_files and os.path.exists(f"/content/drive/MyDrive/RecommendationSystem/{file['name']}")]

    return new_files

folder_id = "*****"
processed_files = []

while True:
    new_files = check_for_new_file(folder_id, processed_files)

    if new_files:
        for file in new_files:
            file_name = file['name']
            file_id = file['id']
            file_path = f"/content/drive/MyDrive/RecommendationSystem/{file_name}"  # Adjust the path as needed

            try:
                with open(file_path, 'r') as f:
                    contents = f.read()
                print("--Processing File--")
                CourseSchedule = pd.read_csv(file_path)
                getRecommendations()
                drive_service.files().delete(fileId=file_id).execute()
                processed_files.append(file_name)
            except FileNotFoundError:
                print(f"Error: File not found: {file_name}")

        # Exit the loop after processing a file
        break

    time.sleep(5)

In [ ]:
BITPlan = np.array ([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Basics of Computing
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #C++
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #JAVA
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #WEB
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #DATA
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Database
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #ADV WEB
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Networks
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #OS
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Algo
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Software
    [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Security

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #discrete
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #حزم احصاء
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #BI

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Calc
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Linear

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Fund
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #MIS

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Project-1 (90 H)

    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #ADV java (E)
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #WEB Server
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #HCI (E)
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Simulation BIT
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #AI (E)
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Special Topics (E)
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #software packs (E)
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Multimedia (E)
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #DB Tools (E)
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Data Mining (E)
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #E-Business
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Moblie
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #E-Learning (E)
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #E-GOV (E)
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #ADV Networks(E)
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #IOT (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Enterprise Dev (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Doc analysis
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #TQM
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #System
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #ITPM
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #E-Payment
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Risk Mang (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #ERP
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #OR (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Semantic (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Information Retrieval (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Knowledge
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Ethics
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Information Rec Mang (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Project2
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Acc (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Micro (E)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Training (90 H)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  #Seminar-Road to Software Industry (45 H)
])

GeneralCourses = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Basics of Arabic

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Basics of English

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Ethics and Humans Values

    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Arabic Languages Skills
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #English Language Skills

    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Entrepreneurship Innovation and Scientific Research
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Life And Practical Skills
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Introduction to Philosophy and Critical Thinking
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Military Science
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #National Culture

    #ELICTIVE

    #FIRST GROUP

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Islam and Contemporary Issues
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Arab-Islamic Civilization
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Jordan: History and Civilization
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Great Books
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Jerusalem

    #SECOND GROUP

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Environmental Culture and Development
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Islamic Culture
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Health Culture
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Legal Culture
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Physical Fitness Culture

    #THIRD GROUP

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Electronic Commerce
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Appreciation of Arts
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #Foreign Language
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] #Special Subject
])


In [ ]:
BITPlan_Dataset = pd.read_csv("/content/BITCoursesDataset.csv")
BITPlan_Dataset["Course Name"] = BITPlan_Dataset["Course Name"].apply(lambda x: re.sub('\xa0', ' ', x))
BITPlan_Names = BITPlan_Dataset["Course Name"].values.tolist()

GeneralCourses_Names = ["Basics of Arabic", "Basics of English", "Ethics and Humans Values", "Arabic Language Skills", "English Language Skills", "Entrepreneurship Innovation and Scientific Research", "Life And Practical Skills", "Introduction to Philosophy and Critical Thinking", "Military Science", "National Culture", "Islam and Contemporary Issues", "Arab-Islamic Civilization", "Jordan: History and Civilization", "Great Books", "Jerusalem", "Environmental Culture and Development", "Islamic Culture", "Health Culture", "Legal Culture", "Physical Fitness Culture", "Electronic Commerce", "Appreciation of Arts", "Foreign Language", "Special Subject"]
NonCreditGeneralCourses = ["Basics of Arabic", "Basics of English","Arabic Language Skills", "English Language Skills"]
ElectiveGeneralCourses_G1 = ["Islam and Contemporary Issues", "Arab-Islamic Civilization", "Jordan: History and Civilization", "Great Books", "Jerusalem"]
ElectiveGeneralCourses_G2 = ["Environmental Culture and Development", "Islamic Culture", "Health Culture", "Legal Culture", "Physical Fitness Culture"]
ElectiveGeneralCourses_G3 = ["Electronic Commerce", "Appreciation of Arts", "Foreign Language", "Special Subject"]

num_BITcourses = BITPlan.shape[0]
popularity_BITvector = np.zeros(num_BITcourses)
popularity_generalvector = np.sum(GeneralCourses, axis = 0)

def dfs(course):
    count = 0
    for i in range(num_BITcourses):
        if BITPlan[i, course] == 1:
            count += 1 + dfs(i)
    return count

for i in range(num_BITcourses):
    popularity_BITvector[i] = dfs(i)

popularity_BITvector = popularity_BITvector.astype(int)

allcourses = BITPlan_Names + GeneralCourses_Names

Popularity_vector = np.concatenate((popularity_BITvector , popularity_generalvector))

Popularity_vector = Popularity_vector/np.max(Popularity_vector)

popularity_dict = dict(zip(allcourses, Popularity_vector))

#Student Class

In [ ]:
studentsarray = []
semester = 'Summer' #Replace by function call?

class StudentCourses:
  def __init__(self, stdname, finished_courses, finishedcoursegrade_Pair):
    self.stdname = stdname
    self.finished_courses = finished_courses
    self.finishedcoursegrade_Pair = finishedcoursegrade_Pair

  def Studenteligible_courses(self):
    student_accumulatedhours = 0
    student_accmajorelective = 0
    isgraduate = False
    self.finished_courses = [s.strip() for s in self.finished_courses]
    self.finishedcoursegrade_Pair = [(x.strip(), y) for x,y in self.finishedcoursegrade_Pair]

    current_vectorMajor = np.zeros(len(BITPlan_Names))
    current_vectorGeneral = np.zeros(len(GeneralCourses_Names))
    popularity_vectorMajor = np.sum(BITPlan, axis=0)
    popularity_vectorGeneral = np.sum(GeneralCourses, axis=0)

    for course in self.finished_courses:
      if course in BITPlan_Names:
        #binary vector for major courses
        index = BITPlan_Names.index(course)
        current_vectorMajor[index] = 1
        student_accumulatedhours = student_accumulatedhours + BITPlan_Dataset["Credit Hours"][index]
        if (BITPlan_Dataset["Course Type"][index] == 'Elective'):
          student_accmajorelective += 3
      else:
        #binary vector for general courses
        index = GeneralCourses_Names.index(course)
        current_vectorGeneral[index] = 1
        if course not in NonCreditGeneralCourses:
          student_accumulatedhours += 3
        if course in ElectiveGeneralCourses_G1:
          self.finished_courses = self.finished_courses + ElectiveGeneralCourses_G1
          self.finished_courses.remove(course)
        elif course in ElectiveGeneralCourses_G2:
          self.finished_courses = self.finished_courses + ElectiveGeneralCourses_G2
          self.finished_courses.remove(course)
        elif course in ElectiveGeneralCourses_G3:
          self.finished_courses = self.finished_courses + ElectiveGeneralCourses_G3
          self.finished_courses.remove(course)

    eligible_vectorMajor = BITPlan @ current_vectorMajor
    eligible_vectorGeneral = GeneralCourses @ current_vectorGeneral

    #Get Courses with no prerequisite
    for index,x in enumerate(BITPlan):
      if np.sum(BITPlan[index]) == 0:
        eligible_vectorMajor[index] = 1
      elif np.sum(BITPlan[index]) > 1:
        if eligible_vectorMajor[index] != np.sum(BITPlan[index]):
          eligible_vectorMajor[index] = 0
        else:
          eligible_vectorMajor[index] = 1

    for index,x in enumerate(GeneralCourses):
      if np.sum(GeneralCourses[index]) == 0:
        eligible_vectorGeneral[index] = 1
      elif np.sum(GeneralCourses[index]) > 1:
        if eligible_vectorGeneral[index] != np.sum(GeneralCourses[index]):
          eligible_vectorGeneral[index] = 0
        else:
          eligible_vectorGeneral[index] = 1

    #Finished 12 elective hours
    if (student_accmajorelective == 12):
      for index in BITPlan_Dataset["Course Type"].index:
        if (BITPlan_Dataset["Course Type"][index] == 'Elective'):
          eligible_vectorMajor[index] = 0

    eligible_courses = [course for i, course in enumerate(BITPlan_Names) if eligible_vectorMajor[i] == 1 and course not in self.finished_courses] + [course for i, course in enumerate(GeneralCourses_Names) if eligible_vectorGeneral[i] == 1 and course not in self.finished_courses]

    if 'Seminar-Road to Software Industry' in eligible_courses and student_accumulatedhours < 45:
      eligible_courses.remove('Seminar-Road to Software Industry')
    if student_accumulatedhours < 90:
      if 'Project 1' in eligible_courses:
        eligible_courses.remove('Project 1')
      if 'Training' in eligible_courses:
        eligible_courses.remove('Training')

    #Is graduate
    if (semester == 'Summer' and student_accumulatedhours >= 120) or (semester == 'FirstorSecond' and student_accumulatedhours >= 111):
      isgraduatearray = eligible_courses.copy()
      prerequisiteBIT = np.sum(BITPlan, axis = 0)
      ispossiblegraduate = True
      def checkCourse(x):
        rows = np.where(BITPlan[:, x] == 1)[0]
        for course in rows:
          if BITPlan_Dataset["Course Type"][index] != 'Elective':
            return False
        return True
      majorelective = 12
      if 'Training' in isgraduatearray:
        isgraduatearray.remove('Training')
      if 'Seminar-Road to Software Industry' in isgraduatearray:
        isgraduatearray.remove('Seminar-Road to Software Industry')
      for course in isgraduatearray:
        if course in BITPlan_Names:
          index = BITPlan_Names.index(course)
          if prerequisiteBIT[index] != 0:
            if checkCourse(index):
              ispossiblegraduate = True
            else:
              ispossiblegraduate = False
          elif BITPlan_Dataset["Course Type"][index] == 'Elective':
            isgraduatearray.remove(course)
            majorelective = student_accmajorelective
        else:
          index = GeneralCourses_Names.index(course)
          if popularity_generalvector[index] != 0:
            ispossiblegraduate = False
      if majorelective != 12:
        while majorelective != 12:
          isgraduatearray.append('Elective Course')
          majorelective +=3
      if course in ElectiveGeneralCourses_G1:
        isgraduatearray = [course for course in isgraduatearray if course not in ElectiveGeneralCourses_G1]
        isgraduatearray.append('Elective Group 1')
      elif course in ElectiveGeneralCourses_G2:
        isgraduatearray = [course for course in isgraduatearray if course not in ElectiveGeneralCourses_G2]
        isgraduatearray.append('Elective Group 2')
      elif course in ElectiveGeneralCourses_G3:
        isgraduatearray = [course for course in isgraduatearray if course not in ElectiveGeneralCourses_G2]
        isgraduatearray.append('Elective Group 3')
      if isgraduatearray and ((semester == 'Summer' and len(isgraduatearray) <= 4) or (semester == 'FirstorSecond' and len(isgraduatearray) <= 7)):
        isgraduate = True

    if len(eligible_courses) != 0 :
      studentsarray.append([self.stdname, self.finishedcoursegrade_Pair, eligible_courses, student_accumulatedhours, isgraduate])

#Data Pre-processing

In [ ]:
BITPlan_CleanDS = pd.DataFrame()
BITPlan_CleanDS['Clean_DataN'] = BITPlan_Dataset['Course Name'].str.lower()
BITPlan_CleanDS['Clean_DataD'] = BITPlan_Dataset['Course Discription'].str.lower()
BITPlan_CleanDS['Clean_DataR'] = BITPlan_Dataset['Course Requirements'].str.lower()

BITPlan_CleanDS['Clean_DataR'] = BITPlan_CleanDS['Clean_DataR'].fillna("")
BITPlan_CleanDS['Clean_DataD'] = BITPlan_CleanDS['Clean_DataD'].fillna("")

BITPlan_CleanDS['Clean_Data'] = BITPlan_CleanDS['Clean_DataN'] + " " + BITPlan_CleanDS['Clean_DataD'] + " " + BITPlan_CleanDS['Clean_DataR']

BITPlan_CleanDS['Clean_Data'] = BITPlan_CleanDS['Clean_Data'].apply(lambda x: re.sub('[^a-zA-Z\s]+','',x))
BITPlan_CleanDS['Clean_Data'] = BITPlan_CleanDS['Clean_Data'].apply(lambda x: re.sub('\s{2,}',' ',x))

#Strat of lemmalzation
lemmatizer = WordNetLemmatizer()

def lemmatize_word(word, pos):
    tag = {'N': wordnet.NOUN,
           'V': wordnet.VERB,
           'R': wordnet.ADV,
           'J': wordnet.ADJ}.get(pos[0].upper(), wordnet.NOUN)
    return lemmatizer.lemmatize(word, tag)

def lemmatize_sentence(sentence):
  tokens = nltk.pos_tag(nltk.word_tokenize(sentence))
  lemmas = [lemmatize_word(word, pos) for word, pos in tokens]
  return ' '.join(lemmas)

BITPlan_CleanDS['Clean_Data'] = BITPlan_CleanDS['Clean_Data'].apply(lemmatize_sentence)

BITPlan_CleanDS['Course_Name'] = BITPlan_Dataset["Course Name"]
BITPlan_CleanDS = BITPlan_CleanDS[['Course_Name','Clean_Data']]

#K-Mean Clustering

In [ ]:
vectorizerC = CountVectorizer(stop_words='english')
featuresC = vectorizerC.fit_transform(BITPlan_CleanDS['Clean_Data'])

#after using the elbow method to determine the optimal number of cluster decided on K = 5
model = KMeans(n_clusters=5,init ='k-means++', max_iter=100 , n_init =7, random_state=0)
model.fit(featuresC)

BITPlan_Dataset['Cluster'] = model.labels_

#Cosine Similarity
vectorizerT = TfidfVectorizer(stop_words='english')
featuresT = vectorizerT.fit_transform(BITPlan_CleanDS['Clean_Data'])

Cosine_Sim = cosine_similarity(featuresT,featuresT)

#Collaborative Filtering

In [ ]:
#preparations
studentsGrades = pd.read_csv('/content/registrationStudents_Dataset.csv',index_col = 0)
studentsGrades = studentsGrades.pivot_table(index='StudentName', columns='Course_Name', values='Grade', aggfunc='first')
studentsGrades = studentsGrades.fillna(0)
missing_courses = ['Database Languages and Tools', 'Semantic Web']
studentsGrades = studentsGrades.reindex(columns=studentsGrades.columns.union(missing_courses), fill_value=0)

In [ ]:
studentsGrades = pd.read_csv('/content/Students_Dataset.csv',index_col = 0)
studentsGrades = studentsGrades.fillna(0)
studentsGrades.columns = [re.sub('\xa0', ' ', x) for x in studentsGrades.columns]

In [ ]:
#preparations
studentsGrades = pd.read_csv('/content/registrationStudents_Dataset.csv',index_col = 0)
studentsGrades = studentsGrades.pivot_table(index='StudentName', columns='Course_Name', values='Grade', aggfunc='first')
studentsGrades = studentsGrades.fillna(0)
missing_courses = ['Database Languages and Tools', 'Semantic Web']
studentsGrades = studentsGrades.reindex(columns=studentsGrades.columns.union(missing_courses), fill_value=0)
studentsGrades.columns = [re.sub('\xa0', ' ', x) for x in studentsGrades.columns]
for student in range(1,len(studentsGrades)):
  finished_courses = []
  grades = []
  for course in range(len(studentsGrades.columns)):
    if(studentsGrades.iloc[student,course] == 'A'):
      studentsGrades.iloc[student,course] = 5
    elif(studentsGrades.iloc[student,course] == 'A-'):
      studentsGrades.iloc[student,course] = 4.75
    elif(studentsGrades.iloc[student,course] == 'B+'):
      studentsGrades.iloc[student,course] = 4.5
    elif(studentsGrades.iloc[student,course] == 'B'):
      studentsGrades.iloc[student,course] = 4
    elif(studentsGrades.iloc[student,course] == 'B-'):
      studentsGrades.iloc[student,course] = 3.75
    elif(studentsGrades.iloc[student,course] == 'C+'):
      studentsGrades.iloc[student,course] = 3.5
    elif(studentsGrades.iloc[student,course] == 'C'):
      studentsGrades.iloc[student,course] = 3
    elif(studentsGrades.iloc[student,course] == 'C-'):
      studentsGrades.iloc[student,course] = 2.75
    elif(studentsGrades.iloc[student,course] == 'D+'):
      studentsGrades.iloc[student,course] = 2.5
    elif(studentsGrades.iloc[student,course] == 'D'):
      studentsGrades.iloc[student,course] = 2
    elif(studentsGrades.iloc[student,course] == 'D-'):
      studentsGrades.iloc[student,course] = 1.75
    elif(studentsGrades.iloc[student,course] == 'F'):
      studentsGrades.iloc[student,course] = 1

    if (studentsGrades.iloc[student,course] != "Pass" and studentsGrades.iloc[student,course] != "Fail" and studentsGrades.iloc[student,course] != 0):
        grades.append([studentsGrades.columns[course],studentsGrades.iloc[student,course]])

    if(studentsGrades.iloc[student,course] != 0 and studentsGrades.iloc[student,course] != 1 and studentsGrades.iloc[student,course] != 1.75 and studentsGrades.iloc[student,course] != "Fail"):
      finished_courses.append(studentsGrades.columns[course])

      #studentsGrades.index[student] = StudentCourses(studentsGrades.index[student], finished_courses(studentsGrades.columns.append(course)))
  student_Object = StudentCourses(studentsGrades.index[student], [re.sub('\xa0', ' ', x) for x in finished_courses], [(re.sub('\xa0', ' ', x), y) for x,y in grades])
  student_Object.Studenteligible_courses()

studentsGrades = studentsGrades.drop("Basics of Computing",axis=1)
studentsGrades = studentsGrades.drop("Project 1",axis=1)
studentsGrades = studentsGrades.drop("Training",axis=1)
studentsGrades = studentsGrades.drop("Seminar-Road to Software Industry",axis=1)
studentsGrades = studentsGrades.drop("Basics of Arabic",axis=1)
studentsGrades = studentsGrades.drop("Basics of English",axis=1)
studentsGrades = studentsGrades.drop("Arabic Language Skills",axis=1)
studentsGrades = studentsGrades.drop("English Language Skills",axis=1)

for student in range(len(studentsGrades)):
  for course in range(len(studentsGrades.columns)):
    if(studentsGrades.iloc[student,course] == 'Pass' or studentsGrades.iloc[student,course] == 'Fail'):
      studentsGrades.iloc[student,course] = 2.5
#Collaborative Filtering
studentsGrades = studentsGrades.astype(float)
studentsGrades = studentsGrades.corr(method='pearson')

def get_collaborativescore(course,grade):
  collaborativescore = studentsGrades[course]*grade
  collaborativescore = collaborativescore.sort_values(ascending=False)

  return 1 / (1 + np.exp(-collaborativescore))

#Combining Data

In [ ]:
studentsdf = pd.DataFrame(studentsarray)
studentsdf = studentsdf.sort_values(by=3, ascending=False)
studentsdf = studentsdf.reset_index(drop=True)

for student,x in enumerate(studentsdf[0]):
  averallcollaborativestdscore = pd.DataFrame()
  eligiblescores = []
  if len(studentsdf[1][student]) != 0:
    for course, grade in studentsdf[1][student]:
      averallcollaborativestdscore = pd.concat([get_collaborativescore(course, grade)], axis=1)
    averallcollaborativestdscore = averallcollaborativestdscore.sum(axis=1)

    for course in studentsdf[2][student]:
      if course in studentsGrades.columns:
        eligiblescores.append(averallcollaborativestdscore[course])
      elif course in studentsdf[2][student]:
        eligiblescores.append(0.0)

    combine = dict(zip(studentsdf[2][student], eligiblescores))

  if len(studentsdf[1][student]) != 0:
    studentsdf[2][student] = combine

#Time Filtering

In [ ]:
CourseSchedule = pd.read_csv('/content/Course_ScheduleSummer.csv')
CourseSchedule = CourseSchedule.fillna(0)
CourseSchedule['Course Name'] = CourseSchedule['Course Name'].apply(lambda x: re.sub('\xa0', ' ', x))

CourseSchedule['Capacity'] = CourseSchedule['Capacity'].astype(int)
CourseSchedule['Registered_Students'] = 0

graduates_conflicting_courses = pd.DataFrame(columns = ['Student', 'Course1', 'Course2'])
popularcourses_timeslots = pd.DataFrame(columns = ['Course Info'])


#Optimization Function (FFF)

In [ ]:
# Optimization Function (FFF)
Final_Filtered_Frame = pd.DataFrame(columns = ['Student', 'Course'])

for index_s,student in enumerate(studentsdf[0][:50]):
    DV_courses = []
    for index_c, course in enumerate(CourseSchedule['Course Name']):
      if course in studentsdf[2][index_s]:
        DV_courses.append((cp.Variable(boolean=True, name = f"DV_{index_c}_{course}"), CourseSchedule.loc[index_c,:]))

    DV_courses = sorted(DV_courses, key=lambda x: x[1]['Course Name'])

    # Popularity objective function:
    objective_popularity = cp.sum([cp.multiply(popularity_dict[course_info['Course Name']] + 1e-6, dv_course) for dv_course, course_info in DV_courses])

    # Collaborative Filtering objective function:
    if len(studentsdf[1][index_s]) != 0:
      objective_cf = cp.sum([cp.multiply(studentsdf[2][index_s][course_info['Course Name']] + 1e-6, dv_course) for dv_course, course_info in DV_courses])
    else:
      objective_cf = cp.sum([cp.multiply(0 + 1e-6, dv_course) for dv_course, course_info in DV_courses])

    #for dv_course1, course_info1 in DV_courses:
      #for dv_course2, course_info2 in DV_courses:
        #if course_info1['Course Name'] in BITPlan_Names and course_info2['Course Name'] in BITPlan_Names and course_info1['Course Name'] != course_info2['Course Name']:
          #objective_sim = cp.sum([cp.multiply(Cosine_Sim[BITPlan_Names.index(course_info1['Course Name']), BITPlan_Names.index(course_info2['Course Name'])] + 1e-6, dv_course) for dv_course, course_info in DV_courses])

    # Constraints
    constraints = []
    chosen_courses = []
    time_slot_decision_vars = {}
    section_decision_vars = {}
    graduates_courses = []
    cluster0_arr = []
    cluster1_arr = []
    cluster2_arr = []
    cluster3_arr = []
    cluster4_arr = []
    university_arr = []
    general_arr = []
    exists = False
    for dv_course, course_info in DV_courses:
      # Credit Hours Constraint: Set up
      if course_info['Credit Hours'] == 3:
        chosen_courses.append(dv_course)

      # One course per time slot and One section per course constraints: Set up
      time_slot = course_info['Time']
      course_name = course_info['Course Name']

      # One Course per Time Slot Constraint:
      if studentsdf[4][index_s] == False:
        if time_slot not in time_slot_decision_vars:
          time_slot_decision_vars[time_slot] = [dv_course]
        else:
          time_slot_decision_vars[time_slot].append(dv_course)
      else:
        if time_slot not in time_slot_decision_vars:
          graduates_courses.append(course_info)
          time_slot_decision_vars[time_slot] = [dv_course]
        elif time_slot != '00:00 - 00:00':
          course2 = next((courseinfo for courseinfo in graduates_courses if courseinfo['Time'] == course_info['Time']), None)
          graduates_conflicting_courses = pd.concat([graduates_conflicting_courses, pd.DataFrame({'Student': student, 'Course1': [course_info], 'Course2': [course2]}, index=[0])], ignore_index=True)
          time_slot_decision_vars[time_slot].append(dv_course)

      # One Section per Course Constraint:
      if course_name not in section_decision_vars:
          section_decision_vars[course_name] = [dv_course]
      else:
          section_decision_vars[course_name].append(dv_course)
      # Capacity constraint:
      if course_info['Capacity'] != 0 and (course_info['Capacity'] - course_info['Registered_Students']) == 0 and dv_course.value == 1:
        for index_p, courseinfo in enumerate(popularcourses_timeslots['Course Info']):
          if course_info == courseinfo:
            popularcourses_timeslots['Number of Students'][index_p] += 1
            exists = True
            break
        if exists == False:
          popularcourses_timeslots = pd.concat([popularcourses_timeslots, pd.DataFrame({'Course Info': [course_info]}, index=[0])], ignore_index=True)
      if course_info['Capacity'] != 0:
        constraints.append(dv_course * (course_info['Capacity'] - course_info['Registered_Students']) >= 0)

      #Soft Constraint: Groups
      if course_info['Course Name'] in BITPlan_Names:
        index = BITPlan_Dataset.loc[BITPlan_Dataset['Course Name'] == course_info['Course Name']].index[0]
        if BITPlan_Dataset.loc[index, 'Cluster'] == 0:
          cluster0_arr.append(dv_course)
        elif BITPlan_Dataset.loc[index, 'Cluster'] == 1:
          cluster1_arr.append(dv_course)
        elif BITPlan_Dataset.loc[index, 'Cluster'] == 2:
          cluster2_arr.append(dv_course)
        elif BITPlan_Dataset.loc[index, 'Cluster'] == 3:
          cluster3_arr.append(dv_course)
        elif BITPlan_Dataset.loc[index, 'Cluster'] == 4:
          cluster4_arr.append(dv_course)
      else:
        if course in GeneralCourses_Names and course not in NonCreditGeneralCourses:
          university_arr.append(dv_course)
        elif course in NonCreditGeneralCourses:
          general_arr.append(dv_course)

    # Credit Hours Constraint
    if semester == 'Summer' and not studentsdf[4][index_s]:
        constraints += [cp.sum(chosen_courses) >= 3, cp.sum(chosen_courses) <= 4]
    elif semester == 'FirstorSecond' and not studentsdf[4][index_s]:
        constraints += [cp.sum(chosen_courses) >= 3, cp.sum(chosen_courses) <= 6]

    # One course per time slot and One section per course constraints:
    for time_slot, dv_courses in time_slot_decision_vars.items():
      constraints.append(cp.sum(dv_courses) <= 1)
    for course_name, dv_courses in section_decision_vars.items():
      constraints.append(cp.sum(dv_courses) <= 1)

    penalty = 10
    constraints += [
          cp.sum(cluster0_arr) <= 1 + penalty,
          cp.sum(cluster1_arr) <= 1 + penalty,
          cp.sum(cluster2_arr) <= 1 + penalty,
          cp.sum(cluster3_arr) <= 1 + penalty,
          cp.sum(cluster4_arr) <= 1 + penalty,
          cp.sum(university_arr) <= 1 + penalty,
          cp.sum(general_arr) <= 1 + penalty
      ]

    objective = cp.Maximize(objective_popularity + objective_cf)
    problem = cp.Problem(objective, constraints)
    problem.solve()

    if problem.status == cp.OPTIMAL:
    # Retrieve the optimal variable values
      optimal_values = [dv_course.value for dv_course in chosen_courses]

    # Print or use the optimal values as needed
      for value, (dv_course, course_info) in zip(optimal_values, DV_courses):
        print(f"{student}, Course: {course_info['Course Name']}, DV value: {value}")
    else:
      print("Optimization problem failed to find an optimal solution.")



In [ ]:
import random

In [ ]:
pd.set_option('display.max_rows', None)
Grades = pd.read_csv('/content/DT.csv')
Grades = Grades[['Student_Num','StudentName','Course_Num','Course_Name','Grade']]
Grades['Grade'] = Grades['Grade'].str.strip()
students = pd.read_csv('/content/Students.csv')
courses = pd.read_csv('/content/Courses.csv')
for index_g in Grades.index:
  for index_s in students.index:
    if Grades['Student_Num'][index_g] == students['Student_Num'][index_s]:
      Grades['StudentName'][index_g] = students['Student_Name'][index_s]

for index_g in Grades.index:
  for index_s in courses.index:
    if Grades['Course_Num'][index_g] == courses['Course_Num'][index_s]:
      Grades['Course_Name'][index_g] = courses['Course_Name'][index_s]

Grades = Grades[Grades['Course_Name'] != '#REF!']

Grades = Grades.reset_index(drop=True)

new_courses = pd.DataFrame({
    'Course_Num': [1902214, 1902390, 1904472, 1904211, 1904487, 1904323, 1904355, 1904357, 1904458, 1904382, 1904486, 1931460, 1905320, 1901364],
    'Course_Name': ['Advanced Java Programming', 'Seminar-Road to Software Industry', 'IT Project Management', 'Mobile Programming', 'E-Payment Systems', 'Knowledge Management Systems', 'E-Learning & Applications', 'E-Government', 'Software Packages', 'Information Retrieval', 'Enterprise Application Development', 'Fundamentals of IoT', 'Artificial Intelligence', 'Advanced Computer Networks'],
    'Student_Num': 0,
    'StudentName': 0,
    'Grade': 0
})

Grades = Grades.append(new_courses, ignore_index=True)
Grades = Grades.reset_index(drop=True)
for index_g in Grades.index:
  Grades['Course_Name'][index_g] = re.sub('\xa0', ' ', Grades['Course_Name'][index_g])

Grades = Grades[['Student_Num','StudentName','Course_Name','Grade']]

for student in range(1, 353):
  coursenames = []
  studentname=''
  for index_g, value_g in Grades['Student_Num'].iteritems():
    if value_g == student:
      coursenames.append(Grades['Course_Name'][index_g])
      studentname = Grades['StudentName'][index_g]

  for course in coursenames:
    prerequisite_courses = []
    stack = [course]  # Stack to store courses and their prerequisites
    visited = set()  # Set to keep track of visited courses to avoid cycles

    while stack:
      current_course = stack.pop()
      prerequisites_indices = []

      if current_course in BITPlan_Names:
        index = BITPlan_Names.index(current_course)
        prerequisites_indices = [index for index, prerequisite in enumerate(BITPlan[index]) if prerequisite == 1]
      else:
        index = GeneralCourses_Names.index(current_course)
        prerequisites_indices = [index for index, prerequisite in enumerate(GeneralCourses[index]) if prerequisite == 1]

      for index in prerequisites_indices:
        if current_course in BITPlan_Names:
          prerequisite_courses.append(BITPlan_Names[index])
        else:
          prerequisite_courses.append(GeneralCourses_Names[index])
        stack.append(prerequisite_courses[-1])  # Add the prerequisite to the stack
        visited.add(prerequisite_courses[-1])  # Mark the prerequisite as visited

    for prerequisite in prerequisite_courses:
      grade = 0
      if prerequisite not in NonCreditGeneralCourses and prerequisite != 'Project 1' and prerequisite != 'Training' and prerequisite != 'Seminar-Road to Software Industry' and prerequisite != 'Basics of Computing':
        grade = random.choice(['A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D'])
      else:
        grade = 'Pass'

      Grades.loc[len(Grades)] = [student, studentname, prerequisite, grade]

<ipython-input-25-67adeed8fa0b>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Grades['StudentName'][index_g] = students['Student_Name'][index_s]
<ipython-input-25-67adeed8fa0b>:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Grades['Course_Name'][index_g] = courses['Course_Name'][index_s]
<ipython-input-25-67adeed8fa0b>:29: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  Grades = Grades.append(new_courses, ignore_index=True)
<ipython-input-25-67adeed8fa0b>:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [ ]:
 if problem.status == cp.OPTIMAL:
    # Retrieve the optimal variable values
    optimal_values = [dv_course.value for dv_course in chosen_courses]

    # Print or use the optimal values as needed
    for value, (dv_course, course_info) in zip(optimal_values, DV_courses):
        print(f"{student}, Course: {course_info['Course Name']}, DV value: {value}")
  else:
    print("Optimization problem failed to find an optimal solution.")